## Foundry Agent Tracing with Application Insights

### Installing Libraries and SDKs

In [ ]:
%pip install azure-ai-projects==2.0.0b2 opentelemetry-sdk==1.42.1 azure-core-tracing-opentelemetry==1.0.0b13 azure-monitor-opentelemetry==1.8.8 opentelemetry-exporter-otlp-proto-http==1.42.1

### Importing Libraries and Utilities

In [ ]:
import os

os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

### Creating the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Instantiating an OpenAI Client

In [ ]:
openai_client = project_client.get_openai_client()

### Fetching the MCP Server Connection Reference

In [ ]:
connection_id = ""

for connection in project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

### Fetching the App Insights Connection String

In [ ]:
connection_string = project_client.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=connection_string)

print("App Insights Connection String: {}".format(connection_string))

### Beginning the Trace

In [ ]:
tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("agent-tracing-scenario"):

    tool = MCPTool(
        server_label = "microsoft_learn_server",
        server_url="https://learn.microsoft.com/api/mcp",
        require_approval="never",
        project_connection_id=connection_id
    )

    agent = project_client.agents.create_version(
        agent_name="MCP-Agent",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="You are an intelligent assistant that can interact with the Microsoft Learn MCP server to provide users with relevant learning resources and information about Microsoft technologies.",
            tools=[tool],
        )
    )

    print(f"Created MCP Agent with ID: {agent.id}")

    # create a conversation to use with the agent
    conversation = openai_client.conversations.create()
    print(f"Created conversation with id: {conversation.id}")

    user_query = "Can you pls help me with the latest ai foundry SDK code samples?" 

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent": {
                "name": "MCP-Agent",
                "type": "agent_reference"
            }
        },
        input=user_query
    )

    print(f"Agent Response: {response.output_text}")
